# Day 1:
- SQL: **Core Retrieval & Logic:** `WHERE` vs. `HAVING`, `CASE WHEN ... THEN ... ELSE ... END`, `COALESCE()`, and `NULLIF()`.
- Pyyhon: **Data Structures & Efficiency:** Hash Maps (Dictionaries) for O(1) lookups, Sets for deduplication, Lists/Arrays (comprehensions, `enumerate`, `zip`, slicing).
- ML and Modelling: **Imbalanced Data:** SMOTE, undersampling, oversampling (avoiding data leakage), class weights. 

## SQL:

In [ ]:
# Imports:
import sqlite3
import pandas as pd

# DB Connection:
conn= sqlite3.connect('oracle_hr.db')

### Question 1:
Write a query to find the department_id and the total number of employees in each department, but only include employees who make a salary greater than $4,000. Furthermore, only return departments that have more than 5 such employees.

In [7]:
pd.read_sql_query(sql= """
select department_id, count(employee_id) as num_emps
from employees
where salary > 4000
group by department_id
having num_emps > 5;
""", con= conn)

,department_id,num_emps
0,50,7
1,80,34
2,100,6


- WHERE vs. HAVING

**Explanation: WHERE applies conditions to individual rows before any grouping occurs. HAVING applies conditions to aggregated groups after the GROUP BY clause has been processed.**

**Core Intuition: Think of WHERE as the bouncer at the front door (filtering individuals before they enter), and HAVING as the manager evaluating tables inside (filtering groups based on their collective behavior).**

- Pros and Cons:
** Pros: Using WHERE early drastically speeds up query performance by reducing the dataset size before expensive grouping operations.**

**Cons: HAVING is computationally heavier; never put a row-level filter in HAVING that could have been handled in WHERE.**

- Edge Cases: Be careful when using HAVING without a GROUP BY clause (which treats the entire table as one group). Also, remember that WHERE cannot evaluate aggregate functions like MAX() or SUM().

- Standard Code Syntax:
```sql
SELECT department_id, COUNT(employee_id) AS num_emps
FROM employees
WHERE salary > 4000
GROUP BY department_id
HAVING COUNT(employee_id) > 5;
```

### Question 2:
Using the HR employees table, write a query that returns the first_name, salary, and a new column named salary_tier.
Categorize the employees as follows:

Less than 5000: 'Low'

Between 5000 and 10000 (inclusive): 'Medium'

Greater than 10000: 'High'.

In [11]:
pd.read_sql_query(sql= """
select first_name, salary,
case when salary < 5000 then 'Low'
    when salary between 5000 and 10000 then 'Medium'
    when salary > 10000 then 'High'
end as salary_tier
from employees;
""", con= conn)


,first_name,salary,salary_tier
0,Steven,24000.0,High
1,Neena,17000.0,High
2,Lex,17000.0,High
3,Alexander,9000.0,Medium
4,Bruce,6000.0,Medium
...,...,...,...
102,Pat,6000.0,Medium
103,Susan,6500.0,Medium
104,Hermann,10000.0,Medium
105,Shelley,12008.0,High


- CASE WHEN

**Explanation: CASE WHEN is SQL's equivalent of IF-THEN-ELSE logic, allowing you to return specific values based on condition evaluations.**

**Core Intuition: It evaluates conditions sequentially. As soon as a condition evaluates to TRUE for a given row, it returns the specified result and skips the remaining checks.**

- Pros and Cons:

**Pros: Incredibly versatile for data bucketing, pivoting, and cleaning data directly in the database.**

**Cons: Long CASE chains can become difficult to read. Evaluating expensive subqueries within a CASE statement can cause massive performance hits.**

- Edge Cases: If you omit the ELSE clause and no WHEN condition is met, SQL will implicitly return NULL. Always ensure your conditions are in the right order to prevent logical overlaps.

- Standard Code Syntax:
```sql
SELECT 
    first_name, 
    salary,
    CASE 
        WHEN salary < 5000 THEN 'Low'
        WHEN salary <= 10000 THEN 'Medium'
        ELSE 'High'
    END AS salary_tier
FROM employees;
```

### Question 3:
In the HR employees table, many employees don't receive a commission, so their commission_pct column is NULL.
Write a query that returns the first_name, salary, commission_pct, and a calculated column called total_compensation.

The total_compensation should be the salary plus the commission amount (salary * commission_pct). However, if commission_pct is NULL, it should be treated as 0. Use COALESCE to handle this.

In [13]:
pd.read_sql_query(sql= """
select first_name, salary, commission_pct,
(salary + (salary * (coalesce(commission_pct, 0)))) as total_compensation
from employees;
""", con= conn)

,first_name,salary,commission_pct,total_compensation
0,Steven,24000.0,NaN,24000.0
1,Neena,17000.0,NaN,17000.0
2,Lex,17000.0,NaN,17000.0
3,Alexander,9000.0,NaN,9000.0
4,Bruce,6000.0,NaN,6000.0
...,...,...,...,...
102,Pat,6000.0,NaN,6000.0
103,Susan,6500.0,NaN,6500.0
104,Hermann,10000.0,NaN,10000.0
105,Shelley,12008.0,NaN,12008.0


### Gold Standard: COALESCE

*   **Explanation:** `COALESCE` is a function that takes a list of arguments and returns the first one that is not `NULL`.
*   **Core Intuition:** It serves as a safety net. When data might be missing (NULL), `COALESCE` provides a default fallback value to ensure calculations or displays don't break.
*   **Pros and Cons:**
    *   *Pros:* Extremely useful for data cleaning on the fly and preventing `NULL` propagation in mathematical operations. It is standard SQL and highly optimized.
    *   *Cons:* Can mask underlying data quality issues if overused. It only checks for `NULL`, not empty strings `''` or zeroes.
*   **Edge Cases:** All arguments inside `COALESCE` should ideally be of the same data type. If they differ, the database engine might throw an error or perform implicit conversions that slow down the query.
*   **Standard Code Syntax:**
```sql
SELECT 
    first_name, 
    salary, 
    commission_pct,
    salary * (1 + COALESCE(commission_pct, 0)) AS total_compensation
FROM employees;

### Question 4:
Using the `employees` table, write a query that returns the `department_id`, the total number of employees in that department, and a column called `dept_comp_tier`.

Here are the requirements:
1.  **Filter:** Exclude employees in department `90` (the Executive department) before doing any math.
2.  **Calculate:** Compute the average total compensation for each department. (Remember: `total_compensation = salary * (1 + COALESCE(commission_pct, 0))`).
3.  **Categorize:** Use a `CASE WHEN` statement on that average compensation to assign a `dept_comp_tier` ('Low' < 5000, 'Medium' <= 10000, 'High' > 10000).
4.  **Filter Again:** Only return departments where the *total number of employees* is greater than 2.

In [21]:
pd.read_sql_query(sql= """
select department_id, count(employee_id) as num_emps, avg(salary * (1 + coalesce(commission_pct, 0))) as avg_comp,
case when avg(salary * (1 + coalesce(commission_pct, 0))) < 5000 then 'Low'
    when avg(salary * (1 + coalesce(commission_pct, 0))) <= 10000 then 'Medium'
    else 'High' 
    end as dept_comp_tier
from employees
where department_id <> 90
group by department_id
having count(employee_id) > 2;
""", con= conn)

,department_id,num_emps,avg_comp,dept_comp_tier
0,30,6,4150.000000,Low
1,50,45,3475.555556,Low
2,60,5,5760.000000,Medium
3,80,34,11092.352941,High
4,100,6,8601.333333,Medium


### Gold Standard: SQL Hybrid (WHERE vs HAVING + CASE + COALESCE)

*   **Explanation:** Combining these functions allows for complex data transformation and layered filtering in a single pass. `WHERE` filters raw rows, `COALESCE` handles missing data during aggregation, `CASE WHEN` creates dynamic categories from those aggregates, and `HAVING` filters the final grouped output.
*   **Core Intuition:** SQL evaluates clauses in a specific order: `FROM` -> `WHERE` -> `GROUP BY` -> `HAVING` -> `SELECT`. Because `SELECT` happens almost last, you must repeat aggregate math in the `CASE` and `HAVING` clauses rather than relying on column aliases (depending on the SQL dialect).
*   **Pros and Cons:**
    *   *Pros:* Pushes heavy lifting to the database layer, which is highly optimized for set-based operations, reducing the amount of data transferred to Python/Pandas.
    *   *Cons:* Repeating complex aggregate formulas (like the average total comp calculation) makes the query verbose and harder to maintain. (In real-world scenarios, a CTE or Subquery is often used to calculate the aggregate first, then apply `CASE` and `HAVING` to keep it clean).
*   **Edge Cases:** If all employees in a department have a `NULL` salary, `AVG()` will return `NULL`, making all `CASE WHEN` conditions fail and resulting in the `ELSE` ('High' in this logic, which would be a bug!). `COALESCE` on the salary column itself might be needed in messy real-world data.
*   **Standard Code Syntax:**
```sql
SELECT 
    department_id, 
    COUNT(employee_id) AS num_emps,
    CASE 
        WHEN AVG(salary * (1 + COALESCE(commission_pct, 0))) < 5000 THEN 'Low'
        WHEN AVG(salary * (1 + COALESCE(commission_pct, 0))) <= 10000 THEN 'Medium'
        ELSE 'High' 
    END AS dept_comp_tier
FROM employees
WHERE department_id IS NOT NULL
GROUP BY department_id
HAVING COUNT(employee_id) > 2;

### Question 5:
We want to evaluate managers and their teams based on the `employees` table. 

Write a query that returns the `manager_id` and the count of their direct reports (`employee_id`). 
Requirements:
1.  **Filter:** Ignore rows where `manager_id` is `NULL`.
2.  **Calculate & Categorize:** Create a column called `team_type`. If the *average* `commission_pct` of a manager's direct reports is greater than `0.10`, call it 'Commission Heavy'. Otherwise, call it 'Standard'. **Crucially**, if an employee has a `NULL` `commission_pct`, it must be treated as `0` when calculating this average.
3.  **Filter Again:** Only include managers who have **3 or more** direct reports.

In [32]:
pd.read_sql_query(sql= """
select manager_id, count(employee_id) direct_reports,
case when avg(coalesce(commission_pct,0)) <= 0.10 then 'Standard'
    else 'Commission Heavy'
end as team_type
from employees
where manager_id is not null
group by manager_id
having count(employee_id) >= 3;
""", con= conn)

,manager_id,direct_reports,team_type
0,100,14,Commission Heavy
1,101,5,Standard
2,103,4,Standard
3,108,5,Standard
4,114,5,Standard
5,120,8,Standard
6,121,8,Standard
7,122,8,Standard
8,123,8,Standard
9,124,8,Standard


### Gold Standard: SQL Hybrid 2 (Advanced Aggregations)

*   **Explanation:** This query combines pre-filtering (`WHERE`), safe aggregation (`COALESCE` inside `AVG`), conditional logic (`CASE WHEN`), and post-filtering (`HAVING`) to analyze hierarchical data.
*   **Core Intuition:** By treating `NULL` commissions as `0` *before* the average is calculated, we prevent the denominator of the average from shrinking. `AVG(commission_pct)` only divides by the number of non-null rows, whereas `AVG(COALESCE(commission_pct, 0))` divides by the total number of rows in the group, giving the true average across the whole team.
*   **Pros and Cons:**
    *   *Pros:* Extremely robust way to handle missing data without skewing mathematical averages.
    *   *Cons:* Deeply nested functions (`AVG(COALESCE(...))`) can be tricky to read. 
*   **Edge Cases:** Watch your inequality operators (`>`, `<`, `>=`, `<=`). "Three or more" always means `>=`.
*   **Standard Code Syntax:**
```sql
SELECT 
    manager_id, 
    COUNT(employee_id) AS direct_reports,
    CASE 
        WHEN AVG(COALESCE(commission_pct, 0)) > 0.10 THEN 'Commission Heavy'
        ELSE 'Standard'
    END AS team_type
FROM employees
WHERE manager_id IS NOT NULL
GROUP BY manager_id
HAVING COUNT(employee_id) >= 3;